# Model Training and Evaluation
This notebook trains the EfficientNet-B0 model, evaluates its performance, and presents results with analysis and visualization.

In [1]:
# 1. Import Libraries
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# ================================
# 1. IMPORT
# ================================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from sklearn.model_selection import train_test_split
import pandas as pd
import os

# ================================
# 2. CONFIG
# ================================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 10

DATA_DIR = 'train/'   # ⚠️ sửa nếu dùng Colab
LABEL_FILE = 'labels.csv'

# ================================
# 3. LOAD DATA
# ================================
df = pd.read_csv(LABEL_FILE)

# đảm bảo filename đúng
df['id'] = df['id'].astype(str) + '.jpg'

# stratified split
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    stratify=df['breed'],
    random_state=42
)

print(f"Train: {len(df_train)}, Val: {len(df_val)}")

# ================================
# 4. DATA GENERATOR
# ================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    df_train,
    directory=DATA_DIR,
    x_col='id',
    y_col='breed',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = val_datagen.flow_from_dataframe(
    df_val,
    directory=DATA_DIR,
    x_col='id',
    y_col='breed',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# $SELECTION_PLACEHOLDER$
NUM_CLASSES = len(train_gen.class_indices)  # DataFrameIterator has no `num_classes`
# or: NUM_CLASSES = df['breed'].nunique()

print("Number of classes:", NUM_CLASSES)
print("Number of classes:", NUM_CLASSES)

# ================================
# 5. BUILD MODEL
# ================================
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # freeze

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ================================
# 6. CALLBACKS
# ================================
checkpoint = ModelCheckpoint(
    'efficientnetb0_best.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

earlystop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-6
)

# ================================
# 7. TRAIN PHASE 1 (FREEZE)
# ================================
print("\n=== TRAIN PHASE 1 (Frozen Base) ===")

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    callbacks=[checkpoint, earlystop, lr_reduce]
)

# ================================
# 8. FINE-TUNING
# ================================
print("\n=== FINE-TUNING ===")

base_model.trainable = True

# chỉ train 20 layer cuối
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ================================
# 9. TRAIN PHASE 2
# ================================
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    callbacks=[checkpoint, earlystop, lr_reduce]
)

# ================================
# 10. LOAD BEST MODEL
# ================================
print("\n=== LOAD BEST MODEL ===")

model = load_model('efficientnetb0_best.h5')

# ================================
# 11. SAVE FINAL MODEL
# ================================
model.save('final_model.h5')

print("\n✅ Training Complete! Model saved as final_model.h5")

Found 8177 validated image filenames belonging to 120 classes.
Found 2045 validated image filenames belonging to 120 classes.


In [ ]:
# 3. Train the Model
# Use callbacks for best model saving and early stopping
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import os

checkpoint = ModelCheckpoint('efficientnetb0_best.h5', monitor='val_accuracy', save_best_only=True, mode='max')
earlystop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

# 3. Train the Model

checkpoint = ModelCheckpoint('efficientnetb0_best.h5', monitor='val_accuracy', save_best_only=True, mode='max')
earlystop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

if 'model' not in globals():
    if os.path.exists('efficientnetb0_best.h5'):
        model = load_model('efficientnetb0_best.h5')
    else:
        num_classes = df_train['breed_encoded'].nunique()
        base = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(img_size, img_size, 3))
        x = base.output
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        x = tf.keras.layers.Dropout(0.2)(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
        model = tf.keras.models.Model(inputs=base.input, outputs=outputs)
        base.trainable = False
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=[checkpoint, earlystop]
)

NameError: name 'model' is not defined

In [ ]:
# 4. Evaluate Model Performance
# Predict on validation set
y_true = df_val['label_encoded'].values
y_pred = model.predict(val_gen)
y_pred_classes = np.argmax(y_pred, axis=1)

print('Classification Report:')
print(classification_report(y_true, y_pred_classes))

print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred_classes))

In [ ]:
# 5. Visualize Training Results
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()

# 6. Analyze and Present Results
- The model's performance is evaluated using accuracy, precision, recall, and F1-score.
- Training and validation curves help diagnose overfitting or underfitting.
- Confusion matrix and classification report provide detailed insights into class-wise performance.

**Conclusion:**
- Summarize findings, discuss potential improvements (e.g., more data, fine-tuning, augmentation), and next steps for deployment or further research.